In [22]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
!pip install -q --upgrade selenium

In [29]:
import shutil
import tempfile
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


# =====================================
# ログイン情報
# =====================================

email = "hosoya.miyo@dpp.co.jp"
password = "Daishin5151"

introduction_text = (
    "プログラミング学習中です！"
    "今はスクレイピングに挑戦しています！"
)


# =====================================
# Chromeの設定
# =====================================

chrome_path = shutil.which("google-chrome")

if chrome_path is None:
    raise RuntimeError(
        "Google Chromeが見つかりません。"
        "最初のインストール用セルを実行してください。"
    )

chrome_options = webdriver.ChromeOptions()
chrome_options.binary_location = chrome_path

chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--no-first-run")
chrome_options.add_argument("--no-default-browser-check")
chrome_options.add_argument("--remote-debugging-port=0")
chrome_options.add_argument("--window-size=1920,1080")

profile_directory = tempfile.mkdtemp(
    prefix="selenium-profile-"
)

chrome_options.add_argument(
    f"--user-data-dir={profile_directory}"
)


# =====================================
# Chromeを起動
# =====================================

chrome_driver = webdriver.Chrome(
    options=chrome_options
)

wait = WebDriverWait(chrome_driver, 60)


try:
    # =====================================
    # アカウント作成ページを開く
    # =====================================

    chrome_driver.get(
        "https://terakoya.sejuku.net/register"
    )

    print("アカウント作成ページを開きました。")


    # =====================================
    # ヘッダーの「ログイン」をクリック
    # =====================================

    header_login_button = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "(//*[normalize-space(.)='ログイン'])[last()]"
            )
        )
    )

    chrome_driver.execute_script(
        "arguments[0].click();",
        header_login_button
    )

    print("ログイン画面を開きました。")


    # =====================================
    # ログイン用メールアドレス入力欄
    # =====================================

    email_input = wait.until(
        EC.visibility_of_element_located(
            (
                By.XPATH,
                "("
                "//input["
                "@type='email' or "
                "@name='email' or "
                "contains(@placeholder, 'メール')"
                "]"
                ")[last()]"
            )
        )
    )

    email_input.clear()
    email_input.send_keys(email)


    # =====================================
    # ログイン用パスワード入力欄
    # =====================================

    password_input = wait.until(
        EC.visibility_of_element_located(
            (
                By.XPATH,
                "(//input[@type='password'])[last()]"
            )
        )
    )

    password_input.clear()
    password_input.send_keys(password)

    print("ログイン情報を入力しました。")


    # =====================================
    # ログインボタンを取得
    # =====================================

    login_elements = chrome_driver.find_elements(
        By.XPATH,
        "//*[normalize-space(.)='ログイン']"
    )

    visible_login_elements = [
        element
        for element in login_elements
        if element.is_displayed()
    ]

    if not visible_login_elements:
        raise RuntimeError(
            "ログインボタンが見つかりませんでした。"
        )

    login_button = visible_login_elements[-1]

    print(
        "クリックする要素：",
        login_button.tag_name,
        login_button.text
    )

    chrome_driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        login_button
    )

    time.sleep(1)

    chrome_driver.execute_script(
        "arguments[0].click();",
        login_button
    )


    # =====================================
    # ホーム画面を待つ
    # =====================================

    wait.until(
        EC.url_contains("/home")
    )

    print("ログインに成功しました。")
    print("ホーム画面に移動しました。")
    print(
        "現在のURL：",
        chrome_driver.current_url
    )

    chrome_driver.save_screenshot(
        "01_ホーム画面.png"
    )


    # =====================================
    # プロフィール画面を開く
    # =====================================

    chrome_driver.get(
        "https://terakoya.sejuku.net/account/profile"
    )

    wait.until(
        EC.url_contains("/account/profile")
    )

    time.sleep(5)

    print("プロフィール画面を開きました。")
    print(
        "現在のURL：",
        chrome_driver.current_url
    )


    # =====================================
    # 「編集」をクリック
    # =====================================

    edit_elements = chrome_driver.find_elements(
        By.XPATH,
        "//*[normalize-space(.)='編集']"
    )

    visible_edit_elements = [
        element
        for element in edit_elements
        if element.is_displayed()
    ]

    if not visible_edit_elements:
        raise RuntimeError(
            "編集ボタンが見つかりませんでした。"
        )

    edit_button = visible_edit_elements[-1]

    print(
        "クリックする編集要素：",
        edit_button.tag_name,
        edit_button.text
    )

    chrome_driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        edit_button
    )

    time.sleep(1)

    chrome_driver.execute_script(
        "arguments[0].click();",
        edit_button
    )

    time.sleep(5)

    print("プロフィールを編集状態にしました。")


    # =====================================
    # 自己紹介入力欄を探す関数
    # =====================================

    def find_introduction_element(driver):
        locator_list = [
            (
                By.XPATH,
                "//textarea[contains(@placeholder, '自己紹介')]"
            ),
            (
                By.XPATH,
                "//textarea[contains(@name, 'intro')]"
            ),
            (
                By.XPATH,
                "//textarea[contains(@id, 'intro')]"
            ),
            (
                By.XPATH,
                "//*[contains(normalize-space(.), '自己紹介')]"
                "/following::textarea[1]"
            ),
            (
                By.TAG_NAME,
                "textarea"
            )
        ]

        for by, selector in locator_list:
            elements = driver.find_elements(
                by,
                selector
            )

            for element in elements:
                if element.is_displayed():
                    return element

        return False


    introduction_input = WebDriverWait(
        chrome_driver,
        60
    ).until(find_introduction_element)

    print(
        "自己紹介欄を見つけました：",
        introduction_input.tag_name
    )


    # =====================================
    # 自己紹介文を入力
    # =====================================

    introduction_input.click()

    # 既存の文章をすべて選択
    introduction_input.send_keys(
        Keys.CONTROL,
        "a"
    )

    # 既存の文章を削除
    introduction_input.send_keys(
        Keys.BACKSPACE
    )

    # 新しい自己紹介文を入力
    introduction_input.send_keys(
        introduction_text
    )

    print("自己紹介文を入力しました。")


    # =====================================
    # 更新ボタンを探す関数
    # =====================================

    def find_update_button(driver):
        xpath = (
            "//*[self::button or @role='button']"
            "[contains(normalize-space(.), '更新') "
            "or contains(normalize-space(.), '保存')]"
        )

        elements = driver.find_elements(
            By.XPATH,
            xpath
        )

        for element in reversed(elements):
            if element.is_displayed():
                return element

        return False


    update_button = WebDriverWait(
        chrome_driver,
        60
    ).until(find_update_button)

    print(
        "更新ボタンを見つけました：",
        update_button.text
    )


    # =====================================
    # 更新ボタンをクリック
    # =====================================

    chrome_driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        update_button
    )

    time.sleep(1)

    chrome_driver.execute_script(
        "arguments[0].click();",
        update_button
    )

    time.sleep(5)


    # =====================================
    # 更新結果を保存
    # =====================================

    chrome_driver.save_screenshot(
        "python-scraping-kadai3-result.png"
    )

    print("自己紹介を更新しました。")
    print("入力内容：", introduction_text)
    print(
        "python-scraping-kadai3-result.pngを保存しました。"
    )


except TimeoutException:
    print("エラー：指定した要素が見つかりませんでした。")
    print("現在のURL：", chrome_driver.current_url)
    print("ページタイトル：", chrome_driver.title)

    body_text = chrome_driver.find_element(
        By.TAG_NAME,
        "body"
    ).text

    print("画面の内容：")
    print(body_text[:3000])

    chrome_driver.save_screenshot(
        "python-scraping-kadai3-error.png"
    )

    print("エラー画面を保存しました。")


except Exception as error:
    print("エラーが発生しました。")
    print("エラーの種類：", type(error).__name__)
    print("エラー内容：", error)
    print("現在のURL：", chrome_driver.current_url)

    body_text = chrome_driver.find_element(
        By.TAG_NAME,
        "body"
    ).text

    print("画面の内容：")
    print(body_text[:3000])

    chrome_driver.save_screenshot(
        "python-scraping-kadai3-error.png"
    )

    print("エラー画面を保存しました。")


finally:
    chrome_driver.quit()

    print("Chromeを終了しました。")

アカウント作成ページを開きました。
ログイン画面を開きました。
ログイン情報を入力しました。
クリックする要素： button ログイン
ログインに成功しました。
ホーム画面に移動しました。
現在のURL： https://terakoya.sejuku.net/home
プロフィール画面を開きました。
現在のURL： https://terakoya.sejuku.net/account/profile
クリックする編集要素： button 編集
プロフィールを編集状態にしました。
自己紹介欄を見つけました： textarea
自己紹介文を入力しました。
更新ボタンを見つけました： 更新する
自己紹介を更新しました。
入力内容： プログラミング学習中です！今はスクレイピングに挑戦しています！
python-scraping-kadai3-result.pngを保存しました。
Chromeを終了しました。
